# Data Preparation: Preprocessing and Feature Engineering

## Data Preprocessing

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [4]:
DATA_PATH = Path("../data/AirQualityUCI.csv")

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    decimal=",",
    na_values=-200
)

empty_columns = [
    column
    for column in ["Unnamed: 15", "Unnamed: 16"]
    if column in df.columns
]

df = (
    df
    .drop(columns=empty_columns)
    .dropna(how="all")
    .copy()
)

df["DateTime"] = pd.to_datetime(
    df["Date"].astype(str)
    + " "
    + df["Time"].astype(str).str.replace(".", ":", regex=False),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

df = (
    df
    .drop(columns=["Date", "Time"])
    .sort_values("DateTime")
    .reset_index(drop=True)
)

In [5]:
target_columns = [
    "CO(GT)",
    "C6H6(GT)",
    "NOx(GT)",
    "NO2(GT)",
]

sensor_features = [
    "PT08.S1(CO)",
    "PT08.S2(NMHC)",
    "PT08.S3(NOx)",
    "PT08.S4(NO2)",
    "PT08.S5(O3)",
]

environment_features = [
    "T",
    "RH",
    "AH",
]

ground_truth_columns = [
    "CO(GT)",
    "NMHC(GT)",
    "C6H6(GT)",
    "NOx(GT)",
    "NO2(GT)",
]

In [10]:
df.shape

(9357, 14)

In [7]:
df.head()

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,DateTime
0,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,2004-03-10 18:00:00
1,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,2004-03-10 19:00:00
2,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,2004-03-10 20:00:00
3,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,2004-03-10 21:00:00
4,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,2004-03-10 22:00:00


In [8]:
df.dtypes

CO(GT)                  float64
PT08.S1(CO)             float64
NMHC(GT)                float64
C6H6(GT)                float64
PT08.S2(NMHC)           float64
NOx(GT)                 float64
PT08.S3(NOx)            float64
NO2(GT)                 float64
PT08.S4(NO2)            float64
PT08.S5(O3)             float64
T                       float64
RH                      float64
AH                      float64
DateTime         datetime64[us]
dtype: object

In [11]:
print("Invalid DateTime:", df["DateTime"].isna().sum())
print("Duplicate DateTime:", df["DateTime"].duplicated().sum())

Invalid DateTime: 0
Duplicate DateTime: 0


In [12]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Start:", df["DateTime"].min())
print("End:", df["DateTime"].max())
print("Sorted:", df["DateTime"].is_monotonic_increasing)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate timestamps:", df["DateTime"].duplicated().sum())

time_gap = df["DateTime"].diff().value_counts().head()
time_gap

Rows: 9357
Columns: 14
Start: 2004-03-10 18:00:00
End: 2005-04-04 14:00:00
Sorted: True
Duplicate rows: 0
Duplicate timestamps: 0


DateTime
0 days 01:00:00    9356
Name: count, dtype: int64